In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split




# Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir="

E0000 00:00:1764862759.132076      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764862759.225614      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [5]:
DATA_DIR = Path("/kaggle/input/stanford-rna-3d-folding/")  # or wherever you uploaded files

# File paths
TRAIN_SEQ_PATH = DATA_DIR / 'train_sequences.csv'
TRAIN_LABELS_PATH = DATA_DIR / 'train_labels.csv'
VAL_SEQ_PATH = DATA_DIR / 'validation_sequences.csv'
VAL_LABELS_PATH = DATA_DIR / 'validation_labels.csv'
TEST_SEQ_PATH = DATA_DIR / 'test_sequences.csv'

In [6]:
train_sequences = pd.read_csv(TRAIN_SEQ_PATH)
train_labels = pd.read_csv(TRAIN_LABELS_PATH)

val_sequences = pd.read_csv(VAL_SEQ_PATH)
val_labels = pd.read_csv(VAL_LABELS_PATH)

test_sequences = pd.read_csv(TEST_SEQ_PATH)

print(f"Train sequences: {train_sequences.shape}, Train labels: {train_labels.shape}")
print(f"Validation sequences: {val_sequences.shape}, Validation labels: {val_labels.shape}")

Train sequences: (844, 5), Train labels: (137095, 6)
Validation sequences: (12, 5), Validation labels: (2515, 123)


In [7]:
# Compute sequence lengths
seq_stats = train_sequences[['target_id', 'sequence']].copy()
seq_stats['seq_len'] = seq_stats['sequence'].str.len()

# Final bucket boundaries (consistent with preprocessing)
buckets = {
    'small': (0, 100),
    'medium': (101, 150),
    'large': (151, 450)
}

def assign_bucket(n):
    for name, (lo, hi) in buckets.items():
        if lo <= n <= hi:
            return name
    return 'large'  

seq_stats['bucket'] = seq_stats['seq_len'].apply(assign_bucket)

# counts
bucket_counts = seq_stats['bucket'].value_counts().reindex(['small','medium','large']).fillna(0).astype(int)
print("Bucket counts (train_sequences):")
print(bucket_counts.to_string())

# Examples per bucket
print("\nExamples per bucket (up to 5 each):")
for b in ['small','medium','large']:
    examples = seq_stats[seq_stats['bucket']==b]['target_id'].tolist()[:5]
    print(f"{b}: {examples}")

# Save for later use
seq_stats.to_csv('/kaggle/working/seq_stats_with_buckets.csv', index=False)
print("\nSaved seq_stats_with_buckets.csv to /kaggle/working/")

# Show distribution quantiles
print("\nSequence length percentiles:")
print(seq_stats['seq_len'].quantile([0.25,0.5,0.75,0.9,0.95]).to_string())


Bucket counts (train_sequences):
bucket
small     667
medium     79
large      98

Examples per bucket (up to 5 each):
small: ['1SCL_A', '1RNK_A', '1RHT_A', '1HLX_A', '1HMH_E']
medium: ['1FFK_9', '1FOQ_A', '1S9S_A', '1YSH_B', '2GO5_A']
large: ['1N34_A', '2A64_A', '2NOQ_A', '2ZJQ_X', '3IYR_A']

Saved seq_stats_with_buckets.csv to /kaggle/working/

Sequence length percentiles:
0.2500    22.0000
0.5000    39.5000
0.7500    86.0000
0.9000   161.4000
0.9500   412.6500


In [8]:
# Build resname map from test_sequences: target_id -> list of residue names (chars)
test_resnames = {}
for _, r in test_sequences.iterrows():
    tid = r['target_id']
    seq = r.get('sequence', "") or ""
    seq = seq.upper().replace('T', 'U')   # normalize T -> U
    # each residue represented by a single character (A/C/G/U or others)
    res_list = list(seq)
    test_resnames[tid] = res_list

# Quick sanity
example_tid = next(iter(test_resnames))
print("Example test_id:", example_tid, "len:", len(test_resnames[example_tid]), "first 10:", test_resnames[example_tid][:10])

Example test_id: R1107 len: 69 first 10: ['G', 'G', 'G', 'G', 'G', 'C', 'C', 'A', 'C', 'A']


In [9]:
OUT_DIR = Path("/kaggle/working/rna_buckets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

#  Bucket boundaries
BUCKETS = {
    'small':  {'min': 0,   'max': 100, 'pad': 100},
    'medium': {'min': 101, 'max': 150, 'pad': 150},
    'large':  {'min': 151, 'max': 450, 'pad': 450}
}

print("Bucket definitions (name: min-max, pad):")
for k,v in BUCKETS.items():
    print(f"  {k}: {v['min']}-{v['max']}  (pad={v['pad']})")

Bucket definitions (name: min-max, pad):
  small: 0-100  (pad=100)
  medium: 101-150  (pad=150)
  large: 151-450  (pad=450)


In [10]:
# Token map and tokenizer
TOKEN_MAP = {'A':0, 'C':1, 'G':2, 'U':3,}  
PAD_TOKEN = 4
VOCAB_SIZE = 5

def tokenize(seq, max_len):
    toks = np.full(max_len, PAD_TOKEN, dtype=np.int32)
    seq = seq.upper().replace('T','U')
    L = min(len(seq), max_len)
    for i, ch in enumerate(seq[:L]):
        toks[i] = TOKEN_MAP.get(ch, PAD_TOKEN)
    return toks, L

In [11]:
# Build mapping: target_id -> {resid: (x,y,z)}
label_coords = defaultdict(dict)

label_cols = [c for c in train_labels.columns if c.startswith('x_')]
x_col = 'x_1' if 'x_1' in train_labels.columns else label_cols[0]
y_col = x_col.replace('x_', 'y_')
z_col = x_col.replace('x_', 'z_')

for _, row in train_labels.iterrows():
    idfull = row['ID']
    if not isinstance(idfull, str):
        continue
    parts = idfull.rsplit('_', 1)
    if len(parts) != 2:
        continue
    tid, resid_str = parts
    try:
        resid = int(resid_str)
    except:
        continue
    x = row.get(x_col, np.nan)
    y = row.get(y_col, np.nan)
    z = row.get(z_col, np.nan)
    if pd.isna(x) or pd.isna(y) or pd.isna(z):
        continue
    label_coords[tid][resid] = (float(x), float(y), float(z))


In [12]:
def preprocess_sequences(df, label_coords=None, is_test=False):
    processed = []

    for _, r in df.iterrows():
        tid = r['target_id']
        seq = r['sequence']
        if not isinstance(seq, str):
            seq = ""
        seq_len = len(seq)

        #  Bucket assignment
        bucket, pad_len = None, None
        for b, rng in BUCKETS.items():
            if rng['min'] <= seq_len <= rng['max']:
                bucket, pad_len = b, rng['pad']
                break
        if bucket is None:
            bucket, pad_len = 'large', BUCKETS['large']['pad']

        # Tokenize
        toks, L = tokenize(seq, pad_len)

        # Attention / mask
        mask = np.zeros((pad_len,), dtype=np.float32)
        mask[:L] = 1.0

        if is_test:
            processed.append({
                "target_id": tid,
                "sequence": seq,
                "seq_len": seq_len,
                "bucket": bucket,
                "tokens": toks,
                "mask": mask,
                "pad_len": pad_len
            })
            continue

        # TRAIN / VAL mode: coordinates + centroid
        coords = np.full((pad_len,3), np.nan, dtype=np.float32)
        tgt_map = label_coords.get(tid, {})
        for resid, (x,y,z) in tgt_map.items():
            idx = resid - 1
            if idx < pad_len:
                coords[idx] = (x,y,z)

        coord_mask = (~np.isnan(coords[:,0])).astype(np.float32)

        if coord_mask.sum() > 0:
            centroid = np.nanmean(coords[coord_mask==1], axis=0)
            coords_centered = coords.copy()
            coords_centered[coord_mask==1] -= centroid
        else:
            centroid = np.zeros((3,), dtype=np.float32)
            coords_centered = coords

        processed.append({
            "target_id": tid,
            "sequence": seq,
            "seq_len": seq_len,
            "bucket": bucket,
            "tokens": toks,
            "mask": coord_mask,
            "coords": coords_centered,
            "centroid": centroid,
            "pad_len": pad_len
        })

    return processed


In [13]:
train_rows = preprocess_sequences(train_sequences, label_coords, is_test=False)
val_rows   = preprocess_sequences(val_sequences,   label_coords, is_test=False)
test_rows  = preprocess_sequences(test_sequences,  is_test=True)

print("Processed sequences:")
print("Train:", len(train_rows))
print("Val:", len(val_rows))
print("Test:", len(test_rows))

Processed sequences:
Train: 844
Val: 12
Test: 12


In [14]:
bucket_stats = {}

for bucket_name in BUCKETS.keys():
    # Train + val combined rows
    bucket_rows = [r for r in train_rows if r['bucket']==bucket_name]
    n = len(bucket_rows)
    if n==0:
        continue

    pad_len = BUCKETS[bucket_name]['pad']
    X = np.stack([r['tokens'] for r in bucket_rows]).astype(np.int32)
    Y = np.stack([r['coords'] for r in bucket_rows]).astype(np.float32)
    M = np.stack([r['mask'] for r in bucket_rows]).astype(np.float32)
    CENT = np.stack([r['centroid'] for r in bucket_rows]).astype(np.float32)
    target_ids = [r['target_id'] for r in bucket_rows]
    seq_lens = np.array([r['seq_len'] for r in bucket_rows], dtype=np.int32)

    Y_clean = np.nan_to_num(Y, nan=0.0)

    # Train/val split
    idxs = np.arange(n)
    train_idx, val_idx = train_test_split(idxs, test_size=0.2, shuffle=True, random_state=42)

    # Split arrays
    X_train, X_val = X[train_idx], X[val_idx]
    Y_train, Y_val = Y_clean[train_idx], Y_clean[val_idx]
    M_train, M_val = M[train_idx], M[val_idx]
    CENT_train, CENT_val = CENT[train_idx], CENT[val_idx]
    ids_train = [target_ids[i] for i in train_idx]
    ids_val   = [target_ids[i] for i in val_idx]

    # Save to disk
    np.save(OUT_DIR / f"X_{bucket_name}_train.npy", X_train)
    np.save(OUT_DIR / f"X_{bucket_name}_val.npy", X_val)
    np.save(OUT_DIR / f"Y_{bucket_name}_train.npy", Y_train)
    np.save(OUT_DIR / f"Y_{bucket_name}_val.npy", Y_val)
    np.save(OUT_DIR / f"M_{bucket_name}_train.npy", M_train)
    np.save(OUT_DIR / f"M_{bucket_name}_val.npy", M_val)
    np.save(OUT_DIR / f"CENT_{bucket_name}_train.npy", CENT_train)
    np.save(OUT_DIR / f"CENT_{bucket_name}_val.npy", CENT_val)
    pd.DataFrame({'target_id': ids_train, 'seq_len': seq_lens[train_idx]}).to_csv(OUT_DIR / f"ids_{bucket_name}_train.csv", index=False)
    pd.DataFrame({'target_id': ids_val,   'seq_len': seq_lens[val_idx]}).to_csv(OUT_DIR / f"ids_{bucket_name}_val.csv", index=False)

    bucket_stats[bucket_name] = {
        'n_total': n,
        'n_train': len(train_idx),
        'n_val': len(val_idx),
        'pad_len': pad_len,
        'X_train_shape': X_train.shape,
        'Y_train_shape': Y_train.shape,
        'M_train_shape': M_train.shape
    }

# Summary
print("\nBucket summary:")
for b, stats in bucket_stats.items():
    print(f"\nBucket: {b}")
    for k,v in stats.items():
        print(f"  {k}: {v}")



Bucket summary:

Bucket: small
  n_total: 667
  n_train: 533
  n_val: 134
  pad_len: 100
  X_train_shape: (533, 100)
  Y_train_shape: (533, 100, 3)
  M_train_shape: (533, 100)

Bucket: medium
  n_total: 79
  n_train: 63
  n_val: 16
  pad_len: 150
  X_train_shape: (63, 150)
  Y_train_shape: (63, 150, 3)
  M_train_shape: (63, 150)

Bucket: large
  n_total: 98
  n_train: 78
  n_val: 20
  pad_len: 450
  X_train_shape: (78, 450)
  Y_train_shape: (78, 450, 3)
  M_train_shape: (78, 450)


In [15]:
def create_tf_dataset(X, Y=None, M=None, batch_size=32, shuffle=False):
    if Y is not None:
        dataset = tf.data.Dataset.from_tensor_slices(
            ({'input_seq': X, 'attention_mask': M}, Y)
        )
    else:
        dataset = tf.data.Dataset.from_tensor_slices(
            ({'input_seq': X, 'attention_mask': M})
        )
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(X), seed=42)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


# Build, Train & Test (Notebook-2)

In [16]:
OUT_DIR = Path("/kaggle/working/rna_buckets")
BATCH_SIZE = 32

train_datasets = {}
val_datasets = {}
test_datasets = {}

for bucket_name in BUCKETS.keys():
    try:
        # Load saved arrays
        X_train = np.load(OUT_DIR / f"X_{bucket_name}_train.npy")
        Y_train = np.load(OUT_DIR / f"Y_{bucket_name}_train.npy")
        M_train = np.load(OUT_DIR / f"M_{bucket_name}_train.npy")

        X_val = np.load(OUT_DIR / f"X_{bucket_name}_val.npy")
        Y_val = np.load(OUT_DIR / f"Y_{bucket_name}_val.npy")
        M_val = np.load(OUT_DIR / f"M_{bucket_name}_val.npy")
        
        # X_test = np.load(OUT_DIR / f"X_{bucket_name}_test.npy")
        # M_test = np.load(OUT_DIR / f"M_{bucket_name}_test.npy")

        train_datasets[bucket_name] = create_tf_dataset(X_train, Y_train, M_train, batch_size=BATCH_SIZE, shuffle=True)
        val_datasets[bucket_name]   = create_tf_dataset(X_val, Y_val, M_val, batch_size=BATCH_SIZE, shuffle=False)
        # test_datasets[bucket_name] = {'X':X_test, 'M': M_test}
    except Exception as e:
        print(f"Did not find data for bucket {bucket_name}: {e}")
        continue

    
print("TensorFlow train/val datasets created for all buckets.")


TensorFlow train/val datasets created for all buckets.


# Transformer

In [17]:
# Final Shared Backbone + Dynamic Heads + Robust Trainer

#  Hyperparameters 
EMBED_DIM = 128
NUM_HEADS = 8
FF_DIM = 256
NUM_BLOCKS = 4
DROPOUT_RATE = 0.2
VOCAB_SIZE = 5        
GRAD_CLIP = 1.0       
LR = 1e-3
MAX_EPOCHS = 30

# --- Buckets (must match preprocessing) ---
BUCKETS = {
    'small': {'pad': 100},
    'medium':{'pad': 150},
    'large': {'pad': 450}
}

# Checkpoint dir
CKPT_DIR = Path("/kaggle/working/rna_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# --- Masked MSE loss ---
def masked_mse(y_true, y_pred, mask):
    mask_exp = tf.expand_dims(mask, axis=-1)  
    diff_sq = tf.square(y_true - y_pred) * mask_exp
    # divide by number of valid coords * 3
    denom = tf.reduce_sum(mask) * 3.0 + 1e-6
    return tf.reduce_sum(diff_sq) / denom

# --- Positional Embedding Layer ---
class PositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)
    def call(self, x):
        # x shape (B, L, D)
        seq_len = tf.shape(x)[1]
        positions = tf.range(seq_len)
        return x + self.pos_emb(positions)

# --- Transformer block (Pre-LN) ---
def transformer_block(x, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, ff_dim=FF_DIM, dropout_rate=DROPOUT_RATE):
    # Pre-LN for stability
    x_norm = layers.LayerNormalization(epsilon=1e-6)(x)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x_norm, x_norm)
    attn = layers.Dropout(dropout_rate)(attn)
    x = x + attn
    # FFN
    x_norm2 = layers.LayerNormalization(epsilon=1e-6)(x)
    ff = layers.Dense(ff_dim, activation='relu')(x_norm2)
    ff = layers.Dense(embed_dim)(ff)
    ff = layers.Dropout(dropout_rate)(ff)
    x = x + ff
    return x

# --- Shared backbone (variable length) ---
def build_shared_backbone(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, max_len=450, num_blocks=NUM_BLOCKS):
    inp_seq = layers.Input(shape=(None,), dtype=tf.int32, name="input_seq")
    inp_mask = layers.Input(shape=(None,), dtype=tf.float32, name="mask_seq")  # kept for compatibility
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inp_seq)   # (B, L, D)
    x = PositionalEmbedding(max_len=max_len, embed_dim=embed_dim)(x)
    for _ in range(num_blocks):
        x = transformer_block(x, embed_dim=embed_dim)
    return models.Model([inp_seq, inp_mask], x, name="shared_backbone")

# --- Dynamic head (accepts any length) ---
def build_head_dynamic(embed_dim=EMBED_DIM):
    inp = layers.Input(shape=(None, embed_dim)) 
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(3)(x)  # x,y,z
    return models.Model(inp, out)

# --- Build backbone and heads ---
backbone = build_shared_backbone(max_len=BUCKETS['large']['pad'], num_blocks=NUM_BLOCKS)
heads = {b: build_head_dynamic(EMBED_DIM) for b in BUCKETS.keys()}

# Print summaries (optional)
print("Backbone summary:")
backbone.summary()
for b, h in heads.items():
    print(f"\nHead '{b}' summary:")
    h.summary()

Backbone summary:


Model: "shared_backbone"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_seq           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │        640 │ input_seq[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 128) │     57,600 │ embedding[0][0]   │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 128) │        256 │ positional_embed… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 128) │    527,488 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, None, 128) │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 128) │          0 │ positional_embed… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 256) │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 128) │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, None, 128) │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 128) │          0 │ add[0][0],        │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 128) │    527,488 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, None, 128) │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 128) │          0 │ add_1[0][0],      │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 256) │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 128) │     32,896 │ dense_2[0][0]   

 Total params: 2,433,920 (9.28 MB)

 Trainable params: 2,433,920 (9.28 MB)

...

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, None, 64)       │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, None, 32)       │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, None, 3)        │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,435 (40.76 KB)

 Trainable params: 10,435 (40.76 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Directory to save checkpoints
CKPT_DIR = Path("/kaggle/working/rna_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

optimizers_dict = {b: tf.keras.optimizers.Adam(learning_rate=LR, clipnorm=GRAD_CLIP) for b in BUCKETS.keys()}

# train/val steps 
def train_step(X_batch, Y_batch, M_batch, head, optimizer):
    with tf.GradientTape() as tape:
        backbone_out = backbone([X_batch['input_seq'], X_batch['attention_mask']], training=True)
        L_out = tf.shape(Y_batch)[1]
        backbone_slice = backbone_out[:, :L_out, :]
        preds = head(backbone_slice, training=True)
        loss = masked_mse(Y_batch, preds, M_batch)

    vars_to_train = backbone.trainable_variables + head.trainable_variables
    grads = tape.gradient(loss, vars_to_train)

    if GRAD_CLIP is not None:
        grads = [tf.clip_by_norm(g, GRAD_CLIP) if g is not None else None for g in grads]

    optimizer.apply_gradients([(g, v) for g, v in zip(grads, vars_to_train) if g is not None])
    return loss


def val_step(X_batch, Y_batch, M_batch, head):
    backbone_out = backbone([X_batch['input_seq'], X_batch['attention_mask']], training=False)
    L_out = tf.shape(Y_batch)[1]
    backbone_slice = backbone_out[:, :L_out, :]
    preds = head(backbone_slice, training=False)
    loss = masked_mse(Y_batch, preds, M_batch)
    return loss


In [19]:
# Trainer with checkpointing and dummy-init for variables
def train_bucket_with_checkpoint(train_ds, val_ds, head, bucket_name, optimizer, epochs=MAX_EPOCHS):
    if optimizer is None:
        raise ValueError("optimizer must be provided for this bucket")
    print(f"\n--- Preparing training for bucket: {bucket_name} ---")

    # Dummy pass to ensure variables created and optimizer built for these vars
    for X_batch, Y_batch in train_ds.take(1):
        M_batch = X_batch['attention_mask']
        # call train_step once to create variables (and optimizer will see variables)
        _ = train_step(X_batch, Y_batch, M_batch, head, optimizer)
        break

    # Setup checkpoint
    ckpt = tf.train.Checkpoint(backbone=backbone, head=head, optimizer=optimizer)
    ckpt_manager = tf.train.CheckpointManager(ckpt, CKPT_DIR / f"{bucket_name}_best", max_to_keep=2)

    best_val_loss = float("inf")

    for epoch in range(epochs):
        print(f"\n=== Epoch {epoch+1}/{epochs} (bucket={bucket_name}) ===")
        # Training loop
        train_loss_sum = 0.0
        n_train = 0
        for X_batch, Y_batch in train_ds:
            M_batch = X_batch['attention_mask']
            loss = train_step(X_batch, Y_batch, M_batch, head, optimizer)
            train_loss_sum += float(loss)    
            n_train += 1
        train_loss = train_loss_sum / max(1, n_train)
        print(f"Train loss: {train_loss:.6f}")

        # Validation loop
        val_loss_sum = 0.0
        n_val = 0
        for X_batch, Y_batch in val_ds:
            M_batch = X_batch['attention_mask']
            loss = val_step(X_batch, Y_batch, M_batch, head)
            val_loss_sum += float(loss)
            n_val += 1
        val_loss = val_loss_sum / max(1, n_val)
        print(f"Val loss: {val_loss:.6f}")

        # Save checkpoint if improved
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            ckpt_save_path = ckpt_manager.save()
            print(f"Validation improved. Checkpoint saved to: {ckpt_save_path}")

    print(f"--- Finished training for bucket: {bucket_name} ---\n")
    return best_val_loss

In [20]:
# Run training for each bucket sequentially
Small_EPOCHS = 7
Medium_EPOCHS = 2
Large_EPOCHS = 2

train_results = {}

train_results['small']  = train_bucket_with_checkpoint(
    train_datasets['small'], val_datasets['small'], heads['small'], "small",
    optimizer=optimizers_dict['small'], epochs=Small_EPOCHS
)
train_results['medium'] = train_bucket_with_checkpoint(
    train_datasets['medium'], val_datasets['medium'], heads['medium'], "medium",
    optimizer=optimizers_dict['medium'], epochs=Medium_EPOCHS
)
train_results['large']  = train_bucket_with_checkpoint(
    train_datasets['large'], val_datasets['large'], heads['large'], "large",
    optimizer=optimizers_dict['large'], epochs=Large_EPOCHS
)

print("All buckets trained. Best val losses:", train_results)


--- Preparing training for bucket: small ---

=== Epoch 1/7 (bucket=small) ===
Train loss: 158.432820
Val loss: 171.891635
Validation improved. Checkpoint saved to: /kaggle/working/rna_checkpoints/small_best/ckpt-1

=== Epoch 2/7 (bucket=small) ===
Train loss: 162.612976
Val loss: 171.729395
Validation improved. Checkpoint saved to: /kaggle/working/rna_checkpoints/small_best/ckpt-2

=== Epoch 3/7 (bucket=small) ===
Train loss: 160.650117
Val loss: 171.778122

=== Epoch 4/7 (bucket=small) ===
Train loss: 161.536504
Val loss: 171.856485

=== Epoch 5/7 (bucket=small) ===
Train loss: 161.237986
Val loss: 171.705392
Validation improved. Checkpoint saved to: /kaggle/working/rna_checkpoints/small_best/ckpt-3

=== Epoch 6/7 (bucket=small) ===
Train loss: 160.773879
Val loss: 171.873450

=== Epoch 7/7 (bucket=small) ===
Train loss: 161.312907
Val loss: 171.703647
Validation improved. Checkpoint saved to: /kaggle/working/rna_checkpoints/small_best/ckpt-4
--- Finished training for bucket: small 

In [21]:
# ==== Load best checkpoints for all buckets ====

bucket_best_ckpts = {
    "small": "/kaggle/working/rna_checkpoints/small_best",
    "medium": "/kaggle/working/rna_checkpoints/medium_best",
    "large": "/kaggle/working/rna_checkpoints/large_best"
}

for b, head in heads.items():
    ckpt_dir = bucket_best_ckpts[b]
    ckpt = tf.train.Checkpoint(backbone=backbone, head=head)
    latest = tf.train.latest_checkpoint(ckpt_dir)
    if latest is None:
        print(f"No checkpoint found for bucket: {b}")
        continue
    ckpt.restore(latest).expect_partial()
    print(f"Loaded best checkpoint for bucket: {b} → {latest}")

Loaded best checkpoint for bucket: small → /kaggle/working/rna_checkpoints/small_best/ckpt-4
Loaded best checkpoint for bucket: medium → /kaggle/working/rna_checkpoints/medium_best/ckpt-2
Loaded best checkpoint for bucket: large → /kaggle/working/rna_checkpoints/large_best/ckpt-2


In [24]:
# ===== Per-sequence MC-dropout inference -> submission.csv =====
import numpy as np
import pandas as pd
from collections import defaultdict

# config
N_SAMPLES = 5                      
MAX_BACKBONE_LEN = BUCKETS['large']['pad']  
PAD_TOKEN = 4

# helper: choose bucket name for a sequence length
def seq_bucket_name(L):
    if L <= 100:        
        return 'small'
    elif L <= 150:      
        return 'medium'
    else:               
        return 'large'

# MC-dropout per sequence with optional chunking if seq > MAX_BACKBONE_LEN
def predict_sequence_mc(tid, seq, n_samples=N_SAMPLES):
    seq = (seq or "").upper().replace('T','U')
    L = len(seq)
    if L == 0:
        return np.zeros((n_samples, 0, 3), dtype=float)

    # Determine which head to use using FULL sequence length
    full_head_name = seq_bucket_name(L)
    full_head = heads[full_head_name]

    all_chunks = []

    for start in range(0, L, MAX_BACKBONE_LEN):
        end = min(start + MAX_BACKBONE_LEN, L)
        seg = seq[start:end]
        seg_len = len(seg)

        toks = np.full((1, seg_len), PAD_TOKEN, dtype=np.int32)
        mask = np.zeros((1, seg_len), dtype=np.float32)
        for i, ch in enumerate(seg):
            toks[0, i] = TOKEN_MAP.get(ch, PAD_TOKEN)
            mask[0, i] = 1.0

        preds_runs = []
        for _ in range(n_samples):
            feats = backbone([toks, mask], training=True)
            pred = full_head(feats, training=True)
            preds_runs.append(pred.numpy()[0, :seg_len, :])

        preds_chunk = np.stack(preds_runs, axis=0)
        all_chunks.append(preds_chunk)

    return np.concatenate(all_chunks, axis=1)




# Build submission rows sequence-by-sequence (no pre-bucketing required here)
submission_rows = []
missing_ids = []
for _, row in test_sequences.iterrows():
    tid = row['target_id']
    seq = row['sequence'] or ""
    seq = seq.upper().replace('T','U')
    L = len(seq)

    if L == 0:
        missing_ids.append(tid)
        continue

    # predict (N_SAMPLES, L, 3)
    P_all = predict_sequence_mc(tid, seq, n_samples=N_SAMPLES)  

    # Sanity: P_all second axis must be >= L
    if P_all.shape[1] < L:
        raise RuntimeError(f"Prediction length {P_all.shape[1]} < sequence length {L} for {tid}")

    # Build rows for residues 1..L
    for i, base in enumerate(seq):
        row_entry = [f"{tid}_{i+1}", base, i+1]
        for k in range(N_SAMPLES):
            x,y,z = P_all[k, i]
            row_entry += [float(x), float(y), float(z)]
        submission_rows.append(row_entry)

# column names expected by Kaggle
columns = ["ID", "resname", "resid"] + [f"{axis}_{n}" for n in range(1, N_SAMPLES+1) for axis in ["x","y","z"]]
submission_df = pd.DataFrame(submission_rows, columns=columns)
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv; rows:", len(submission_df))
if missing_ids:
    print("Warning: sequences with empty sequence skipped:", missing_ids)


# ===== Quick validator (basic checks) =====
def validate_submission(sub_df, test_df):
    # expected number of rows
    expected_rows = sum(len(s or "") for s in test_df['sequence'])
    produced_rows = len(sub_df)
    print("Expected rows:", expected_rows, "Produced rows:", produced_rows)

    # check IDs
    expected_ids = { f"{tid}_{i+1}" for _, r in test_df.iterrows() for i, _ in enumerate((r['sequence'] or "")) }
    produced_ids = set(sub_df['ID'].astype(str).tolist())
    missing = expected_ids - produced_ids
    extra = produced_ids - expected_ids
    print("Missing IDs:", len(missing))
    if len(missing) < 20:
        print(sorted(list(missing)))
    print("Extra IDs:", len(extra))
    if len(extra) < 20:
        print(sorted(list(extra)))

    # columns and NaNs
    required_cols = columns
    missing_cols = [c for c in required_cols if c not in sub_df.columns]
    print("Missing columns:", missing_cols)
    nan_counts = sub_df.isna().sum()
    print("NaNs per column (non-zero):")
    print(nan_counts[nan_counts>0])

    missing_ids = missing  
    extra_ids = extra     
    
    print("Example missing IDs:", list(missing_ids)[:20])
    print("Example extra IDs:", list(extra_ids)[:20])


validate_submission(submission_df, test_sequences)


Saved submission.csv; rows: 2515
Expected rows: 2515 Produced rows: 2515
Missing IDs: 602
Extra IDs: 2397
Missing columns: []
NaNs per column (non-zero):
Series([], dtype: int64)
Example missing IDs: ['R1190_650', 'R1190_550', 'R1190_270', 'R1190_596', 'R1190_582', 'R1190_657', 'R1190_339', 'R1190_416', 'R1190_445', 'R1190_137', 'R1190_708', 'R1190_148', 'R1190_661', 'R1190_427', 'R1190_170', 'R1190_288', 'R1190_468', 'R1190_380', 'R1190_227', 'R1190_160']
Example extra IDs: ['R1128_155', 'R1108_42', 'R1136_325', 'R1128_118', 'R1128_231', 'R1126_147', 'R1136_111', 'R1136_231', 'R1126_171', 'R1138_718', 'R1156_24', 'R1138_398', 'R1138_454', 'R1126_25', 'R1116_115', 'R1138_411', 'R1138_415', 'R1138_365', 'R1138_233', 'R1138_49']
